### Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
import matplotlib.patches as mpatches
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from shapely.geometry import Point
from pyproj import Transformer
from matplotlib.lines import Line2D
from sklearn.neighbors import BallTree

In [2]:
base_path = "/mnt/raid/emotional_data_raquel/fulldata_mine"

___________

## Check cum_dist quality

In [3]:
results = []

for root, dirs, files in os.walk(base_path):
    if "alldata.csv" in files:
        file_path = os.path.join(root, "alldata.csv")
        
        parts = root.split(os.sep)
        
        try:
            sub = [p for p in parts if p.startswith("sub-")][0]
        except:
            sub = "unknown_sub"
            
        try:
            ses = [p for p in parts if p.startswith("ses-")][0]
        except:
            ses = "unknown_ses"
        
        # Clean session name (important for grouping)
        session_clean = ses.replace("ses-", "").replace("ses_", "")
        
        label = f"{sub} | {session_clean}"
        
        try:
            df = pd.read_csv(file_path)
            
            has_column = "cum_dist" in df.columns
            
            if has_column:
                non_null = df["cum_dist"].notna().sum()
                has_values = non_null > 0
                percent_valid = (non_null / len(df)) * 100
            else:
                has_values = False
                percent_valid = 0
            
            results.append({
                #"participant_session": label,
                "participant": sub,
                "session": session_clean,
                "has_cum_dist_column": has_column,
                "has_values": has_values,
                "valid_%": round(percent_valid, 2)
            })
        
        except Exception as e:
            results.append({
                #"participant_session": label,
                "participant": sub,
                "session": session_clean,
                "has_cum_dist_column": False,
                "has_values": False,
                "valid_%": 0,
                "error": str(e)
            })

results_df = pd.DataFrame(results)
print("\nFull results:")
print(results_df)


Full results:
   participant    session  has_cum_dist_column  has_values  valid_%
0    sub-OE020   Norrebro                False       False      0.0
1    sub-OE020   Nordhavn                 True        True    100.0
2    sub-OE005   Nordhavn                False       False      0.0
3    sub-OE005   Hellerup                 True        True    100.0
4    sub-OE018   Hellerup                 True        True    100.0
5    sub-OE012  Norreport                False       False      0.0
6    sub-OE021   Norrebro                False       False      0.0
7    sub-OE021   Hellerup                False       False      0.0
8    sub-OE015  Norreport                 True        True    100.0
9    sub-OE002   Hellerup                 True        True    100.0
10   sub-OE022   Norrebro                False       False      0.0
11   sub-OE022  Norreport                 True        True    100.0
12   sub-OE022   Nordhavn                 True        True    100.0
13   sub-OE009   Norrebro        

In [4]:
# Count valid sessions per path
valid_sessions = results_df[results_df["has_values"] == True]
count_per_path = valid_sessions.groupby("session").size().reset_index(name="n_valid_sessions")

print("\nValid sessions per path:")
print(count_per_path)

# Also print invalid
invalid_sessions = results_df[results_df["has_values"] == False]
count_invalid = invalid_sessions.groupby("session").size().reset_index(name="n_invalid_sessions")

print("\nInvalid sessions per path:")
print(count_invalid)


Valid sessions per path:
     session  n_valid_sessions
0   Hellerup                 4
1   Nordhavn                 5
2   Norrebro                 1
3  Norreport                 4

Invalid sessions per path:
     session  n_invalid_sessions
0   Hellerup                   2
1   Nordhavn                   3
2   Norrebro                   4
3  Norreport                   2


# Define segments

In [5]:
segments_dict = {
    "Norreport": ("OE004", [
        (0, 360, "U+M"),
        (360, 410, "U+M"),
        (410, 548, "U+N+M"),
        (548, 767, "N"),
        (767, 884, "U"),
        (884, 990, "U+N"),
        (990, 1149, "U"),
        (1149, 1214, "U+M"),
        (1214, 1400, "U+N+M")
    ]),

    "Norrebro": ("OE023", [
        (421, 562, "U+M"),
        (562, 962, "U+M"),
        (962, 1110, "U+N+M"),
        (1110, 1390, "N"),
        (1390, 1600, "U+N")
    ]),

    "Nordhavn": ("OE010", [
        (0, 225, "U+M"),
        (225, 488, "U+N+M"),
        (488, 658, "U+M"),
        (658, 819, "U"),
        (819, 895, "U+N"),
        (895, 1044, "U+N"),
        (1044, 1227, "U+N"),
        (1227, 1400, "U+M")
    ]),
    
    "Hellerup": ("OE002", [
        (0, 198, "U+M"),
        (198, 395, "U+N"),
        (395, 683, "U+N"),
        (683, 1161, "U"),
        (1161, 1260, "U+N"),
        (1260, 1500, "N")
    ])
}

## Helper Functions

In [6]:
def video_to_real(video_seconds, video_start_time):
    return video_start_time + pd.Timedelta(seconds=video_seconds)

def time_to_distance(df, timestamp):
    row = df.iloc[(df['time'] - timestamp).abs().argsort()[:1]]
    return row['cum_dist'].values[0]

def is_valid_session(df):
    return ("cum_dist" in df.columns) and (df["cum_dist"].notna().any())

## Valid sessions

In [7]:
valid_sessions = []

for participant in os.listdir(base_path):

    participant_path = os.path.join(base_path, participant)

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)

        if not os.path.isdir(session_path):
            continue

        file_path = os.path.join(session_path, "alldata.csv")

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        if is_valid_session(df):
            session_name = session.split("-")[1]
            participant_id = participant.replace("sub-", "")

            valid_sessions.append({
                "participant": participant_id,
                "session": session_name,
                "path": file_path
            })

valid_sessions_df = pd.DataFrame(valid_sessions)

print("\nVALID SESSIONS:")
print(valid_sessions_df)


VALID SESSIONS:
   participant    session                                               path
0        OE020   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
1        OE005   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
2        OE018   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
3        OE015  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
4        OE002   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
5        OE022  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
6        OE022   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
7        OE009   Nordhavn  /mnt/raid/emotional_data_raquel/fulldata_mine/...
8        OE019   Hellerup  /mnt/raid/emotional_data_raquel/fulldata_mine/...
9        OE023   Norrebro  /mnt/raid/emotional_data_raquel/fulldata_mine/...
10       OE023  Norreport  /mnt/raid/emotional_data_raquel/fulldata_mine/...
11       OE023   Nordhavn  /mnt/raid/emotional_data_raquel/

# Building Typology Map (per path)

In [8]:
typology_map = []

for path_name, (ref_participant, segments) in segments_dict.items():

    print(f"\nProcessing reference for {path_name}...")

    # find reference session
    ref_row = valid_sessions_df[
        (valid_sessions_df["participant"] == ref_participant) &
        (valid_sessions_df["session"] == path_name)
    ]

    if ref_row.empty:
        print(f"⚠️ Missing reference for {path_name}")
        continue

    ref_path = ref_row.iloc[0]["path"]
    df_ref = pd.read_csv(ref_path, parse_dates=["time"]).sort_values("time")

    video_start_time = df_ref["time"].iloc[0]

    for start_sec, end_sec, typology in segments:

        t_start = video_to_real(start_sec, video_start_time)
        t_end   = video_to_real(end_sec, video_start_time)

        d_start = time_to_distance(df_ref, t_start)
        d_end   = time_to_distance(df_ref, t_end)

        typology_map.append({
            "path": path_name,
            "lowerbound": d_start,
            "higherbound": d_end,
            "typology": typology
        })

typology_df = pd.DataFrame(typology_map)
typology_df = typology_df.sort_values(["path", "lowerbound"]).reset_index(drop=True)
typology_df = typology_df[typology_df["higherbound"] > typology_df["lowerbound"]]

print("\nTYPOLOGY MAP:")
print(typology_df)


Processing reference for Norreport...

Processing reference for Norrebro...

Processing reference for Nordhavn...

Processing reference for Hellerup...

TYPOLOGY MAP:
         path  lowerbound  higherbound typology
0    Hellerup        19.0         52.4      U+M
1    Hellerup        52.4        136.2      U+N
2    Hellerup       136.2        439.2      U+N
3    Hellerup       439.2       1096.7        U
4    Hellerup      1096.7       1249.3      U+N
5    Hellerup      1249.3       1414.9        N
6    Nordhavn         0.0         37.9      U+M
7    Nordhavn        37.9        218.3    U+N+M
8    Nordhavn       218.3        415.9      U+M
9    Nordhavn       415.9        626.8        U
10   Nordhavn       626.8        730.5      U+N
11   Nordhavn       730.5        825.1      U+N
12   Nordhavn       825.1       1023.0      U+N
13   Nordhavn      1023.0       1200.8      U+M
14   Norrebro         2.9         83.4      U+M
15   Norrebro        83.4        414.7      U+M
16   Norrebro   

## Apply to all valid sessions

In [9]:
def assign_typology(df, path_name, typology_df):

    path_rows = typology_df[typology_df["path"] == path_name]

    def find_typology(dist):
        match = path_rows[
            (path_rows["lowerbound"] <= dist) &
            (dist < path_rows["higherbound"])
        ]
        if match.empty:
            return np.nan
        return match.iloc[0]["typology"]

    df["typology"] = df["cum_dist"].apply(find_typology)

    return df

In [10]:
# save updated files
for _, row in valid_sessions_df.iterrows():

    participant = row["participant"]
    session = row["session"]
    file_path = row["path"]

    print(f"\nApplying typology → {participant} | {session}")

    df = pd.read_csv(file_path, parse_dates=["time"])

    df = assign_typology(df, session, typology_df)

    # Get folder path
    folder = os.path.dirname(file_path)

    # Define NEW filename (no replacement)
    output_path = os.path.join(folder, "alldata_with_typology.csv")

    df.to_csv(output_path, index=False)

print("\nDONE")


Applying typology → OE020 | Nordhavn

Applying typology → OE005 | Hellerup

Applying typology → OE018 | Hellerup

Applying typology → OE015 | Norreport

Applying typology → OE002 | Hellerup

Applying typology → OE022 | Norreport

Applying typology → OE022 | Nordhavn

Applying typology → OE009 | Nordhavn

Applying typology → OE019 | Hellerup

Applying typology → OE023 | Norrebro

Applying typology → OE023 | Norreport

Applying typology → OE023 | Nordhavn

Applying typology → OE004 | Norreport

Applying typology → OE010 | Nordhavn

DONE


In [11]:
all_data = [] 
# ----------------------------- 
# # LOAD ALL FILES 
# # ----------------------------- 
for participant in os.listdir(base_path): 
    participant_path = os.path.join(base_path, participant) 
    if not os.path.isdir(participant_path): 
        continue 
    
    for session in os.listdir(participant_path): 
        session_path = os.path.join(participant_path, session) 
        
        if not os.path.isdir(session_path): 
            continue 
        
        file_path = os.path.join(session_path, "alldata_with_typology.csv") 
        
        if not os.path.exists(file_path): 
            continue 
        
        try: 
            df = pd.read_csv(file_path) 
            df["participant_id"] = participant.replace("sub-", "") 
            df["session_id"] = session.replace("ses-", "").replace("ses_", "") 
            
            all_data.append(df) 
        
        except Exception as e: 
            print(f"⚠️ Error loading {file_path}: {e}") 
            
# Combine everything 
df_all = pd.concat(all_data, ignore_index=True)

/tmp/ipykernel_523830/2664496864.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(all_data, ignore_index=True)


In [12]:
# =====================================================
# TYPOLOGY MAPS
# VALID SESSIONS ONLY (WITH cum_dist)
# =====================================================

# -----------------------------
# LOAD VALID SESSIONS
# -----------------------------
all_data = []

for _, row in valid_sessions_df.iterrows():

    try:

        df = pd.read_csv(
            row["path"],
            parse_dates=["time"]
        )

        # -----------------------------------------
        # ASSIGN TYPOLOGY
        # -----------------------------------------
        df = assign_typology(
            df,
            row["session"],
            typology_df
        )

        # -----------------------------------------
        # PARTICIPANT + SESSION IDS
        # -----------------------------------------
        df["participant_id"] = row["participant"]

        df["session_id"] = row["session"]

        all_data.append(df)

        print(
            f"✓ Loaded → "
            f"{row['participant']} | {row['session']}"
        )

    except Exception as e:

        print(
            f"⚠️ Failed → "
            f"{row['participant']} | "
            f"{row['session']} | {e}"
        )

# -----------------------------
# COMBINE
# -----------------------------
df_all = pd.concat(
    all_data,
    ignore_index=True
)

# =====================================================
# FIXED TYPOLOGY ORDER + COLORS
# =====================================================

typology_order = [
    "N",
    "U",
    "U+M",
    "U+N",
    "U+N+M"
]

typology_colors = {

    "N": "#2ca25f",
    "U": "#7f7f7f",
    "U+M": "#e34a33",
    "U+N": "#41b6c4",
    "U+N+M": "#225ea8"
}

colors = [
    typology_colors[t]
    for t in typology_order
]

cmap = mcolors.ListedColormap(colors)

# =====================================================
# OUTPUT DIRECTORY
# =====================================================

output_dir = (
    "/mnt/raid/emotional_data_raquel/"
    "typology_maps"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

# =====================================================
# LOOP THROUGH SESSIONS
# =====================================================

for (participant, session), df_sess in df_all.groupby(
    ["participant_id", "session_id"]
):

    print(
        f"\nPlotting → "
        f"{participant} | {session}"
    )

    # =================================================
    # COPY
    # =================================================

    df_sess = df_sess.copy()

    # =================================================
    # REMOVE INVALID GPS
    # =================================================

    df_sess = df_sess.dropna(
        subset=[
            "longitude_corrected",
            "latitude_corrected"
        ]
    )

    if len(df_sess) == 0:

        print(
            "⚠️ No valid GPS rows"
        )

        continue

    # =================================================
    # SORT
    # =================================================

    if "time" in df_sess.columns:

        df_sess = df_sess.sort_values(
            "time"
        )

    # =================================================
    # REMOVE DUPLICATES
    # =================================================

    df_sess = df_sess.drop_duplicates(
        subset=[
            "longitude_corrected",
            "latitude_corrected",
            "typology"
        ]
    )

    # =================================================
    # CREATE GEODATAFRAME
    # =================================================

    gdf = gpd.GeoDataFrame(
        df_sess,
        geometry=gpd.points_from_xy(
            df_sess["longitude_corrected"],
            df_sess["latitude_corrected"]
        ),
        crs="EPSG:4326"
    )

    # =================================================
    # TYPOLOGY CATEGORIES
    # =================================================

    gdf["typologies"] = pd.Categorical(
        gdf["typology"],
        categories=typology_order,
        ordered=True
    )

    # =================================================
    # PROJECT
    # =================================================

    gdf_web = gdf.to_crs(
        epsg=3857
    )

    # =================================================
    # REMOVE NaN TYPOLOGIES
    # =================================================

    gdf_web = gdf_web.dropna(
        subset=["typologies"]
    )

    if len(gdf_web) == 0:

        print(
            f"⚠️ No typology data → "
            f"{participant} | {session}"
        )

        continue

    # =================================================
    # AXIS FORMAT
    # =================================================

    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True
    )

    def transform_x(x, _):

        lon, _ = transformer.transform(
            x,
            0
        )

        return f"{lon:.3f}°"

    def transform_y(y, _):

        _, lat = transformer.transform(
            0,
            y
        )

        return f"{lat:.3f}°"

    # =================================================
    # FIGURE
    # =================================================

    fig, ax = plt.subplots(
        figsize=(15, 10)
    )

    # =================================================
    # PLOT TYPOLOGIES
    # =================================================

    gdf_web.plot(
        ax=ax,
        column="typologies",
        cmap=cmap,
        legend=True,
        markersize=90,
        alpha=0.90
    )

    # =================================================
    # BASEMAP
    # =================================================

    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron
    )

    # =================================================
    # LEGEND
    # =================================================

    leg = ax.get_legend()

    if leg is not None:

        leg.set_title(
            "Typologies"
        )

    # =================================================
    # AXIS FORMAT
    # =================================================

    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(transform_x)
    )

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(transform_y)
    )

    ax.set_xlabel(
        "Longitude",
        fontsize=12
    )

    ax.set_ylabel(
        "Latitude",
        fontsize=12
    )

    # =================================================
    # TITLE
    # =================================================

    ax.set_title(
        f"Participant {participant} "
        f"— {session}",
        fontsize=14
    )

    # =================================================
    # SAVE
    # =================================================

    save_path = os.path.join(
        output_dir,
        f"sub-{participant}_ses-{session}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"✓ Saved → {save_path}"
    )

print("\nDONE")

✓ Loaded → OE020 | Nordhavn
✓ Loaded → OE005 | Hellerup
✓ Loaded → OE018 | Hellerup
✓ Loaded → OE015 | Norreport
✓ Loaded → OE002 | Hellerup
✓ Loaded → OE022 | Norreport
✓ Loaded → OE022 | Nordhavn
✓ Loaded → OE009 | Nordhavn
✓ Loaded → OE019 | Hellerup
✓ Loaded → OE023 | Norrebro
✓ Loaded → OE023 | Norreport
✓ Loaded → OE023 | Nordhavn
✓ Loaded → OE004 | Norreport
✓ Loaded → OE010 | Nordhavn

Plotting → OE002 | Hellerup
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps/sub-OE002_ses-Hellerup.png

Plotting → OE004 | Norreport
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps/sub-OE004_ses-Norreport.png

Plotting → OE005 | Hellerup
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps/sub-OE005_ses-Hellerup.png

Plotting → OE009 | Nordhavn
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps/sub-OE009_ses-Nordhavn.png

Plotting → OE010 | Nordhavn
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps/sub-OE010_ses-Nordhavn.png

Plotting → OE015 | Norreport
✓ Saved → /m

______

# Add typologies to participants who dont have cum_dist

In [13]:
# =========================================================
# BUILD REFERENCE TYPOLOGY MAPS
# =========================================================

reference_maps = {}

for path_name, (ref_participant, _) in segments_dict.items():

    print(f"\nBuilding reference map for {path_name}...")

    ref_row = valid_sessions_df[
        (valid_sessions_df["participant"] == ref_participant) &
        (valid_sessions_df["session"] == path_name)
    ]

    if ref_row.empty:

        print(f"⚠️ Missing reference for {path_name}")
        continue

    ref_path = ref_row.iloc[0]["path"]

    df_ref = pd.read_csv(
        ref_path,
        parse_dates=["time"]
    )

    # -----------------------------------------------------
    # ASSIGN TYPOLOGY USING EXISTING cum_dist
    # -----------------------------------------------------
    df_ref = assign_typology(
        df_ref,
        path_name,
        typology_df
    )

    # -----------------------------------------------------
    # KEEP VALID REFERENCE ROWS
    # -----------------------------------------------------
    df_ref = df_ref.dropna(
        subset=[
            "latitude_corrected",
            "longitude_corrected",
            "typology"
        ]
    )

    reference_maps[path_name] = df_ref[
        [
            "latitude_corrected",
            "longitude_corrected",
            "typology"
        ]
    ].copy()

    print(f"✓ Stored {len(df_ref)} reference points")


Building reference map for Norreport...
✓ Stored 1194 reference points

Building reference map for Norrebro...
✓ Stored 1101 reference points

Building reference map for Nordhavn...
✓ Stored 1390 reference points

Building reference map for Hellerup...
✓ Stored 1370 reference points


In [14]:
# =========================================================
# TYPOLOGY ASSIGNMENT VIA SPATIAL MATCHING
# =========================================================

def assign_typology_spatial(
    df,
    session_name,
    reference_maps,
    max_distance_m=10
):

    print(f"Spatial assignment → {session_name}")

    # -----------------------------------------------------
    # GET REFERENCE MAP
    # -----------------------------------------------------
    if session_name not in reference_maps:

        print(f"⚠️ No reference map for {session_name}")

        df["typology_raquel"] = np.nan
        df["typology_match_distance_m"] = np.nan

        return df

    ref_df = reference_maps[session_name]

    # -----------------------------------------------------
    # VALID PARTICIPANT GPS
    # -----------------------------------------------------
    valid_mask = (
        df["original_latitude"].notna()
        &
        df["original_longitude"].notna()
    )

    typologies = [np.nan] * len(df)
    distances_output = [np.nan] * len(df)

    if valid_mask.sum() == 0:

        print("⚠️ No valid GPS rows")

        df["typology_raquel"] = typologies
        df["typology_match_distance_m"] = distances_output

        return df

    # -----------------------------------------------------
    # BUILD SPATIAL TREE
    # -----------------------------------------------------
    ref_coords = np.radians(
        ref_df[
            ["latitude_corrected", "longitude_corrected"]
        ].values
    )

    tree = BallTree(
        ref_coords,
        metric="haversine"
    )

    # -----------------------------------------------------
    # PARTICIPANT COORDS
    # -----------------------------------------------------
    participant_coords = np.radians(
        df.loc[
            valid_mask,
            ["original_latitude", "original_longitude"]
        ].values
    )

    # -----------------------------------------------------
    # NEAREST MATCH
    # -----------------------------------------------------
    distances, indices = tree.query(
        participant_coords,
        k=1
    )

    earth_radius_m = 6371000

    distances_m = (
        distances.flatten()
        * earth_radius_m
    )

    indices = indices.flatten()

    # -----------------------------------------------------
    # ASSIGN TYPOLOGY
    # -----------------------------------------------------
    valid_indices = df.index[valid_mask]

    for row_idx, dist_m, ref_idx in zip(
        valid_indices,
        distances_m,
        indices
    ):

        distances_output[row_idx] = dist_m

        if dist_m <= max_distance_m:

            typologies[row_idx] = (
                ref_df.iloc[ref_idx]["typology"]
            )

    # -----------------------------------------------------
    # SAVE RESULTS
    # -----------------------------------------------------
    df["typology_raquel"] = typologies

    df["typology_match_distance_m"] = distances_output

    matched = pd.Series(typologies).notna().sum()

    print(
        f"✓ Assigned typology to "
        f"{matched}/{len(df)} rows"
    )

    return df

In [15]:
# =========================================================
# APPLY TO FILES WITHOUT cum_dist
# =========================================================

for participant in os.listdir(base_path):

    participant_path = os.path.join(
        base_path,
        participant
    )

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(
            participant_path,
            session
        )

        if not os.path.isdir(session_path):
            continue

        file_path = os.path.join(
            session_path,
            "alldata.csv"
        )

        if not os.path.exists(file_path):
            continue

        print(f"\nProcessing → {participant} | {session}")

        try:

            df = pd.read_csv(
                file_path,
                parse_dates=["time"]
            )

        except Exception as e:

            print(f"⚠️ Failed loading file: {e}")
            continue

        # -------------------------------------------------
        # SKIP FILES THAT ALREADY HAVE cum_dist
        # -------------------------------------------------
        if (
            "cum_dist" in df.columns
            and df["cum_dist"].notna().any()
        ):

            print("✓ cum_dist exists → skipping")

            continue

        print("→ No cum_dist found")

        # -------------------------------------------------
        # SPATIAL TYPOLOGY ASSIGNMENT
        # -------------------------------------------------
        df = assign_typology_spatial(
            df=df,
            session_name=session.replace("ses-", ""),
            reference_maps=reference_maps,
            max_distance_m=10
        )

        # -------------------------------------------------
        # SAVE
        # -------------------------------------------------
        output_path = os.path.join(
            session_path,
            "alldata_with_typology.csv"
        )

        df.to_csv(
            output_path,
            index=False
        )

        print(f"✓ Saved → {output_path}")

print("\nDONE")


Processing → sub-OE020 | ses-Norrebro
→ No cum_dist found
Spatial assignment → Norrebro
✓ Assigned typology to 1214/1576 rows
✓ Saved → /mnt/raid/emotional_data_raquel/fulldata_mine/sub-OE020/ses-Norrebro/alldata_with_typology.csv

Processing → sub-OE020 | ses-Nordhavn
✓ cum_dist exists → skipping

Processing → sub-OE005 | ses-Nordhavn
→ No cum_dist found
Spatial assignment → Nordhavn
✓ Assigned typology to 1003/1383 rows
✓ Saved → /mnt/raid/emotional_data_raquel/fulldata_mine/sub-OE005/ses-Nordhavn/alldata_with_typology.csv

Processing → sub-OE005 | ses-Hellerup
✓ cum_dist exists → skipping

Processing → sub-OE018 | ses-Hellerup
✓ cum_dist exists → skipping

Processing → sub-OE012 | ses-Norreport
→ No cum_dist found
Spatial assignment → Norreport
⚠️ No valid GPS rows
✓ Saved → /mnt/raid/emotional_data_raquel/fulldata_mine/sub-OE012/ses-Norreport/alldata_with_typology.csv

Processing → sub-OE021 | ses-Norrebro
→ No cum_dist found
Spatial assignment → Norrebro
✓ Assigned typology to 90

In [16]:
# -----------------------------
# FIXED TYPOLOGY ORDER + COLORS
# -----------------------------
typology_order = ["N", "U", "U+M", "U+N", "U+N+M", "NaN"]

typology_colors = {
    "N": "#2ca25f",
    "U": "#7f7f7f",
    "U+M": "#e34a33",
    "U+N": "#41b6c4",
    "U+N+M": "#225ea8",
    "NaN": "#000000"
}

colors = [typology_colors[t] for t in typology_order]

cmap = mcolors.ListedColormap(colors)

# -----------------------------
# OUTPUT DIRECTORY
# -----------------------------
output_dir = "/mnt/raid/emotional_data_raquel/typology_maps_raquel"

os.makedirs(output_dir, exist_ok=True)

# =====================================================
# LOOP THROUGH ALL FILES
# =====================================================

for participant in os.listdir(base_path):

    participant_path = os.path.join(
        base_path,
        participant
    )

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(
            participant_path,
            session
        )

        if not os.path.isdir(session_path):
            continue

        # -------------------------------------------------
        # LOAD FILE
        # -------------------------------------------------
        file_path = os.path.join(
            session_path,
            "alldata_with_typology.csv"
        )

        if not os.path.exists(file_path):
            continue

        print(f"\nPlotting → {participant} | {session}")

        try:

            df_sess = pd.read_csv(file_path)

        except Exception as e:

            print(f"⚠️ Could not load file: {e}")

            continue

        # -------------------------------------------------
        # ONLY PLOT SESSIONS WITHOUT cum_dist
        # -------------------------------------------------
        if (
            "cum_dist" in df_sess.columns
            and df_sess["cum_dist"].notna().any()
        ):

            print("✓ Has cum_dist → skipping")

            continue

        # -------------------------------------------------
        # REQUIRE TYPOLOGY_RAQUEL
        # -------------------------------------------------
        if "typology_raquel" not in df_sess.columns:

            print("⚠️ Missing typology_raquel")

            continue

        # =====================================================
        # MANUAL GAP FIXES
        # =====================================================

        session_clean = session.replace("ses-", "")
        
        # -------------------------------------------------
        # NORREBRO
        # Fill NaN gaps related to N
        # -------------------------------------------------
        if session_clean == "Norrebro":

            typ = df_sess["typology_raquel"].copy()

            # ---------------------------------------------
            # CASE 1:
            # Fill NaN runs BETWEEN N segments
            # ---------------------------------------------
            nan_mask = typ.isna()

            groups = (
                nan_mask != nan_mask.shift()
            ).cumsum()

            for grp in groups[nan_mask].unique():

                idx = typ.index[groups == grp]

                start_i = idx.min()
                end_i = idx.max()

                if (
                    start_i > 0
                    and end_i < len(typ) - 1
                ):

                    prev_val = typ.iloc[start_i - 1]
                    next_val = typ.iloc[end_i + 1]

                    if (
                        prev_val == "N"
                        and next_val == "N"
                    ):

                        typ.loc[idx] = "N"

                        print(
                            f"✓ Filled internal "
                            f"N gap ({len(idx)} rows)"
                        )

            # ---------------------------------------------
            # CASE 2:
            # If N is followed by long NaN stretch,
            # extend N forward until next category
            # appears
            # ---------------------------------------------
            typ_values = typ.values

            for i in range(len(typ_values) - 1):

                # Find start of NaN run after N
                if (
                    typ_values[i] == "N"
                    and pd.isna(typ_values[i + 1])
                ):

                    j = i + 1

                    # Move through NaNs
                    while (
                        j < len(typ_values)
                        and pd.isna(typ_values[j])
                    ):

                        j += 1

                    # Only fill if another category appears later
                    if j < len(typ_values):

                        typ.iloc[i + 1:j] = "N"

                        print(
                            f"✓ Extended N through "
                            f"{j - (i + 1)} NaN rows"
                        )

            df_sess["typology_raquel"] = typ

        # -------------------------------------------------
        # HELLERUP
        # Fill NaN runs between U segments
        # -------------------------------------------------
        if session_clean == "Hellerup":

            typ = df_sess["typology_raquel"].copy()

            nan_mask = typ.isna()

            groups = (
                nan_mask != nan_mask.shift()
            ).cumsum()

            for grp in groups[nan_mask].unique():

                idx = typ.index[groups == grp]

                start_i = idx.min()
                end_i = idx.max()

                if (
                    start_i > 0
                    and end_i < len(typ) - 1
                ):

                    prev_val = typ.iloc[start_i - 1]
                    next_val = typ.iloc[end_i + 1]

                    # Fill only if surrounded by U
                    if (
                        prev_val == "U"
                        and next_val == "U"
                    ):

                        typ.loc[idx] = "U"

                        print(
                            f"✓ Filled {len(idx)} "
                            f"Hellerup rows with U"
                        )

            df_sess["typology_raquel"] = typ

        # -------------------------------------------------
        # SAVE UPDATED FILE
        # -------------------------------------------------
        df_sess.to_csv(
            file_path,
            index=False
        )

        # -------------------------------------------------
        # REMOVE INVALID COORDS
        # -------------------------------------------------
        df_sess = df_sess.dropna(
            subset=[
                "original_longitude",
                "original_latitude"
            ]
        )

        if len(df_sess) == 0:

            print("⚠️ No valid rows")

            continue

        # -------------------------------------------------
        # CONVERT NaN TYPOLOGIES INTO STRING
        # -------------------------------------------------
        df_sess["typology_plot"] = (
            df_sess["typology_raquel"]
            .fillna("NaN")
        )

        # -------------------------------------------------
        # SORT
        # -------------------------------------------------
        if "time" in df_sess.columns:

            df_sess = df_sess.sort_values("time")

        # -------------------------------------------------
        # REMOVE DUPLICATES
        # -------------------------------------------------
        df_sess = df_sess.drop_duplicates(
            subset=[
                "original_longitude",
                "original_latitude",
                "typology_plot"
            ]
        )

        # -------------------------------------------------
        # CREATE GEODATAFRAME
        # -------------------------------------------------
        gdf = gpd.GeoDataFrame(
            df_sess,
            geometry=gpd.points_from_xy(
                df_sess["original_longitude"],
                df_sess["original_latitude"]
            ),
            crs="EPSG:4326"
        )

        # -------------------------------------------------
        # TYPOLOGY ORDERING
        # -------------------------------------------------
        gdf["typologies"] = pd.Categorical(
            gdf["typology_plot"],
            categories=typology_order,
            ordered=True
        )

        # -------------------------------------------------
        # PROJECT
        # -------------------------------------------------
        gdf_web = gdf.to_crs(epsg=3857)

        # -------------------------------------------------
        # AXIS FORMAT
        # -------------------------------------------------
        transformer = Transformer.from_crs(
            "EPSG:3857",
            "EPSG:4326",
            always_xy=True
        )

        def transform_x(x, _):

            lon, _ = transformer.transform(x, 0)

            return f"{lon:.3f}°"

        def transform_y(y, _):

            _, lat = transformer.transform(0, y)

            return f"{lat:.3f}°"

        # -------------------------------------------------
        # PLOT
        # -------------------------------------------------
        fig, ax = plt.subplots(figsize=(15, 10))

        gdf_web.plot(
            ax=ax,
            column="typologies",
            cmap=cmap,
            legend=True,
            markersize=35,
            alpha=0.9
        )

        # -------------------------------------------------
        # BASEMAP
        # -------------------------------------------------
        ctx.add_basemap(
            ax,
            source=ctx.providers.CartoDB.Positron
        )

        # -------------------------------------------------
        # LEGEND
        # -------------------------------------------------
        leg = ax.get_legend()

        if leg is not None:

            leg.set_title("Typologies")

        # -------------------------------------------------
        # AXES
        # -------------------------------------------------
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(transform_x)
        )

        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(transform_y)
        )

        ax.set_xlabel(
            "Longitude",
            fontsize=12
        )

        ax.set_ylabel(
            "Latitude",
            fontsize=12
        )

        ax.set_title(
            f"Participant "
            f"{participant.replace('sub-', '')} "
            f"— "
            f"{session.replace('ses-', '')}",
            fontsize=14
        )

        # -------------------------------------------------
        # SAVE FIGURE
        # -------------------------------------------------
        save_path = os.path.join(
            output_dir,
            f"{participant}_{session}_raquel.png"
        )

        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.close()

        print(f"✓ Saved → {save_path}")

print("\nDONE")


Plotting → sub-OE020 | ses-Norrebro
✓ Filled internal N gap (220 rows)


✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps_raquel/sub-OE020_ses-Norrebro_raquel.png

Plotting → sub-OE020 | ses-Nordhavn
✓ Has cum_dist → skipping

Plotting → sub-OE005 | ses-Nordhavn
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps_raquel/sub-OE005_ses-Nordhavn_raquel.png

Plotting → sub-OE005 | ses-Hellerup
✓ Has cum_dist → skipping

Plotting → sub-OE018 | ses-Hellerup
✓ Has cum_dist → skipping

Plotting → sub-OE012 | ses-Norreport
⚠️ No valid rows

Plotting → sub-OE021 | ses-Norrebro
✓ Filled internal N gap (180 rows)
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps_raquel/sub-OE021_ses-Norrebro_raquel.png

Plotting → sub-OE021 | ses-Hellerup
✓ Filled 20 Hellerup rows with U
✓ Filled 19 Hellerup rows with U
✓ Filled 3 Hellerup rows with U
✓ Filled 36 Hellerup rows with U
✓ Saved → /mnt/raid/emotional_data_raquel/typology_maps_raquel/sub-OE021_ses-Hellerup_raquel.png

Plotting → sub-OE015 | ses-Norreport
✓ Has cum_dist → skipping

Plotting → sub-OE002 | ses

# Introduce Danish municipal zoning dataset

In [17]:
# --------------------------------------------------
# LOAD MUNICIPAL ZONING
# --------------------------------------------------
landuse = gpd.read_file("/mnt/raid/emotional_data_raquel/hovedstaden_landuse.gpkg")

# project to same CRS as GPS
landuse = landuse.to_crs(25832)

# --------------------------------------------------
# TRANSLATION
# --------------------------------------------------
translation = {
    "Andet": "Other",
    "Blandet bolig og erhverv": "Mixed Residential and Commercial",
    "Boligområde": "Residential Area",
    "Centerområde": "Urban Center",
    "Erhvervsområde": "Commercial / Industrial Area",
    "Landområde": "Rural Area",
    "Område til offentlige formål": "Public Service Area",
    "Rekreativt område": "Recreational Area",
    "Sommerhusområde": "Vacation Housing Area",
    "Tekniske anlæg": "Technical Infrastructure"
}

landuse["landuse_en"] = (
    landuse["anvendelsegenerel"]
    .map(translation)
)

/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/geopandas/io/file.py:364: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore")
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/geopandas/io/file.py:368: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore", utc=True)
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/geopandas/io/file.py:364: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  as_dt = pd.to_datetime(df[k], errors="ignore")
/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/geopandas/io/file.py:368: F

In [18]:
# --------------------------------------------------
# DEFINE LANDUSE COLORS (same as QGIS)
# --------------------------------------------------
landuse_colors = {
    "Other": "#27aae1",
    "Mixed Residential and Commercial": "#b7d933",
    "Residential Area": "#f2a900",
    "Urban Center": "#e67c73",
    "Commercial / Industrial Area": "#ec268f",
    "Rural Area": "#00e640",
    "Public Service Area": "#6fdc5b",
    "Recreational Area": "#00e6d2",
    "Vacation Housing Area": "#001dff",
    "Technical Infrastructure": "#d96ad9"
}

# Put typologies and municipality zones together

In [20]:
# =====================================================
# PROCESS ALL SESSIONS
# USING:
# alldata_with_typology.csv
# =====================================================

output_dir = (
    "/mnt/raid/emotional_data_raquel/"
    "typology_landuse_maps"
)

os.makedirs(output_dir, exist_ok=True)

# =====================================================
# LOOP THROUGH PARTICIPANTS
# =====================================================

for participant in os.listdir(base_path):
    
    participant_path = os.path.join(
        base_path,
        participant
        )

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(
            participant_path,
            session
        )

        if not os.path.isdir(session_path):
            continue
            
        if participant == "sub-OE002" and session == "ses-Hellerup":

            # =================================================
            # LOAD FILE
            # =================================================

            file_path = os.path.join(
                session_path,
                "alldata_with_typology.csv"
            )

            if not os.path.exists(file_path):

                print(
                    f"⚠️ Missing file → "
                    f"{participant} | {session}"
                )

                continue

            print(
                f"\nProcessing → "
                f"{participant} | {session}"
            )

            # =================================================
            # LOAD CSV
            # =================================================

            try:

                df = pd.read_csv(
                    file_path,
                    parse_dates=["time"]
                )

            except Exception as e:

                print(
                    f"⚠️ Could not load file: {e}"
                )

                continue

            # =================================================
            # CLEAN SESSION NAME
            # =================================================

            session_clean = (
                session
                .replace("ses-", "")
                .replace("ses_", "")
            )

            # =================================================
            # TYPOLOGY COLUMN
            # =================================================

            if "typology_raquel" in df.columns:

                typology_col = "typology_raquel"

            elif "typology" in df.columns:

                typology_col = "typology"

            else:

                print(
                    "⚠️ No typology column found"
                )

                continue

            # =================================================
            # CHOOSE GPS SOURCE
            # =================================================

            has_corrected = (

                "longitude_corrected" in df.columns
                and

                "latitude_corrected" in df.columns
                and

                df["longitude_corrected"]
                .notna()
                .any()

                and

                df["latitude_corrected"]
                .notna()
                .any()
            )

            if has_corrected:

                lon_col = "longitude_corrected"
                lat_col = "latitude_corrected"

                print(
                    "✓ Using corrected coordinates"
                )

            else:

                lon_col = "original_longitude"
                lat_col = "original_latitude"

                print(
                    "✓ Using original coordinates"
                )

            # =================================================
            # REMOVE INVALID GPS
            # =================================================

            df = df.dropna(
                subset=[lon_col, lat_col]
            )

            if len(df) == 0:

                print(
                    "⚠️ No valid GPS rows"
                )

                continue

            # =================================================
            # REMOVE DUPLICATES
            # =================================================

            df = df.drop_duplicates(
                subset=[
                    lon_col,
                    lat_col,
                    typology_col
                ]
            )

            # =================================================
            # CREATE GEODATAFRAME
            # =================================================

            gdf = gpd.GeoDataFrame(
                df,
                geometry=gpd.points_from_xy(
                    df[lon_col],
                    df[lat_col]
                ),
                crs="EPSG:4326"
            )

            # =================================================
            # LANDUSE SPATIAL JOIN
            # =================================================

            gdf_landuse = gdf.to_crs(25832)

            # -------------------------------------------------
            # REMOVE OLD LANDUSE COLUMNS
            # -------------------------------------------------

            gdf_landuse = gdf_landuse.drop(
                columns=[
                    "landuse_en",
                    "landuse_en_left",
                    "landuse_en_right"
                ],
                errors="ignore"
            )

            # -------------------------------------------------
            # SPATIAL JOIN
            # -------------------------------------------------

            gdf_landuse = gpd.sjoin(
                gdf_landuse,
                landuse[
                    ["landuse_en", "geometry"]
                ],
                how="left",
                predicate="within"
            )

            gdf_landuse = gdf_landuse.drop(
                columns=["index_right"],
                errors="ignore"
            )

            # =================================================
            # SAVE UPDATED CSV
            # =================================================

            output_csv = os.path.join(
                session_path,
                "alldata_with_typology_landuse.csv"
            )

            df_out = pd.DataFrame(
                gdf_landuse.drop(
                    columns="geometry"
                )
            )

            df_out.to_csv(
                output_csv,
                index=False
            )

            print(
                f"✓ Saved CSV → {output_csv}"
            )

            # =================================================
            # PREPARE FOR PLOTTING
            # =================================================

            gdf_plot = gdf_landuse.to_crs(
                epsg=3857
            )

            # -------------------------------------------------
            # FORCE NaNs TO BECOME PLOTTABLE CATEGORY
            # -------------------------------------------------

            gdf_plot["typology_plot"] = (
                gdf_plot[typology_col]
                .astype(object)
            )

            gdf_plot.loc[
                gdf_plot["typology_plot"].isna(),
                "typology_plot"
            ] = "NaN"

            # DEBUG
            # print(
            #     gdf_plot["typology_plot"]
            #     .value_counts(dropna=False)
            # )

            # =================================================
            # SMALL LANDUSE SUBSET
            # =================================================

            bounds = (
                gdf_landuse
                .buffer(150)
                .total_bounds
            )

            landuse_small = landuse.cx[
                bounds[0]:bounds[2],
                bounds[1]:bounds[3]
            ]

            landuse_web = (
                landuse_small
                .to_crs(epsg=3857)
            )

            landuse_web["plot_color"] = (
                landuse_web["landuse_en"]
                .map(landuse_colors)
                .fillna("#cccccc")
            )

            # =================================================
            # FIGURE
            # =================================================

            fig, ax = plt.subplots(
                figsize=(14, 12)
            )

            # -------------------------------------------------
            # LANDUSE
            # -------------------------------------------------

            landuse_web.plot(
                ax=ax,
                color=landuse_web["plot_color"],
                alpha=0.20,
                linewidth=1,
                zorder=1
            )

            # =================================================
            # PLOT EACH TYPOLOGY SEPARATELY
            # THIS MAKES NaNs SHOW
            # =================================================

            for typology in typology_order:

                subset = gdf_plot[
                    gdf_plot["typology_plot"]
                    == typology
                ]

                if len(subset) == 0:
                    continue

                subset.plot(
                    ax=ax,
                    color=typology_colors[typology],
                    markersize=40,
                    alpha=0.95,
                    zorder=3
                )

            # =================================================
            # ZOOM
            # =================================================

            xmin, ymin, xmax, ymax = (
                gdf_plot.total_bounds
            )

            padding = 100

            ax.set_xlim(
                xmin - padding,
                xmax + padding
            )

            ax.set_ylim(
                ymin - padding,
                ymax + padding
            )

            # =================================================
            # BASEMAP
            # =================================================

            ctx.add_basemap(
                ax,
                source=ctx.providers.CartoDB.Positron,
                alpha=0.45,
                zorder=0
            )

            # =================================================
            # LANDUSE LEGEND POSITION
            # =================================================

            if session_clean.lower() == "nordhavn":

                landuse_loc = "upper left"

            else:

                landuse_loc = "upper right"

            # =================================================
            # LANDUSE LEGEND
            # =================================================

            landuse_patches = [

                mpatches.Patch(
                    color=color,
                    label=label,
                    alpha=0.5
                )

                for label, color
                in landuse_colors.items()
            ]

            legend_landuse = ax.legend(
                handles=landuse_patches,
                title="Municipality zones",
                loc=landuse_loc,
                fontsize=8,
                frameon=True
            )

            ax.add_artist(
                legend_landuse
            )

            # =================================================
            # TYPOLOGY LEGEND
            # =================================================

            typology_lines = [

                Line2D(
                    [0],
                    [0],
                    color=color,
                    lw=4,
                    label=label
                )

                for label, color
                in typology_colors.items()
            ]

            legend_typology = ax.legend(
                handles=typology_lines,
                title="Participant typologies",
                loc="lower right",
                fontsize=8,
                frameon=True
            )

            ax.add_artist(
                legend_typology
            )

            # =================================================
            # TITLE
            # =================================================

            ax.set_title(
                #f"Participant {participant} "
                f"{session_clean} Typologies and Landuse",
                fontsize=16,
                pad=10
            )

            ax.set_xticks([])
            ax.set_yticks([])

            # =================================================
            # SAVE FIGURE
            # =================================================

            save_path = os.path.join(
                output_dir,
                f"{session_clean}_typologies_land.png"
            )

            plt.savefig(
                save_path,
                dpi=300,
                bbox_inches="tight"
            )

            plt.close()

            print(
                f"✓ Saved plot → {save_path}"
            )

    print("\nDONE")

    # OE023 | Norrebro
    # OE015 | Norreport
    # OE010 | Nordhavn
    # OE002 | Hellerup  


DONE

DONE

DONE

DONE

DONE

DONE

Processing → sub-OE002 | ses-Hellerup
✓ Using corrected coordinates
✓ Saved CSV → /mnt/raid/emotional_data_raquel/fulldata_mine/sub-OE002/ses-Hellerup/alldata_with_typology_landuse.csv
✓ Saved plot → /mnt/raid/emotional_data_raquel/typology_landuse_maps/Hellerup_typologies_land.png

DONE

DONE

DONE

DONE

DONE

DONE

DONE

DONE

DONE

DONE

DONE


In [43]:
# --------------------------------------------------
# ADD TYPOLOGY + LANDUSE TO EACH SESSION
# --------------------------------------------------
for _, row in valid_sessions_df.iterrows():

    participant = row["participant"]
    session = row["session"]
    file_path = row["path"]

    print(f"\nProcessing → {participant} | {session}")

    df = pd.read_csv(file_path, parse_dates=["time"])

    # --------------------------------------------------
    # ASSIGN TYPOLOGY
    # --------------------------------------------------
    df = assign_typology(df, session, typology_df)

    # --------------------------------------------------
    # CREATE GEO DATAFRAME
    # --------------------------------------------------
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude_corrected"],
            df["latitude_corrected"]
        ),
        crs="EPSG:4326"
    )

    # project to same CRS as zoning
    gdf = gdf.to_crs(25832)

    # --------------------------------------------------
    # SPATIAL JOIN WITH MUNICIPAL ZONING
    # --------------------------------------------------
    gdf = gpd.sjoin(
        gdf,
        landuse[["landuse_en", "geometry"]],
        how="left",
        predicate="within"
    )

    # remove duplicate spatial join columns
    gdf = gdf.drop(columns=["index_right"], errors="ignore")

    # --------------------------------------------------
    # SAVE UPDATED CSV
    # --------------------------------------------------
    folder = os.path.dirname(file_path)

    output_path = os.path.join(
        folder,
        "alldata_with_typology_landuse.csv"
    )

    # back to regular dataframe
    df_out = pd.DataFrame(gdf.drop(columns="geometry"))

    df_out.to_csv(output_path, index=False)

print("\nDONE")


Processing → OE020 | Nordhavn

Processing → OE005 | Hellerup

Processing → OE018 | Hellerup

Processing → OE015 | Norreport

Processing → OE002 | Hellerup

Processing → OE022 | Norreport

Processing → OE022 | Nordhavn

Processing → OE009 | Nordhavn

Processing → OE019 | Hellerup

Processing → OE023 | Norrebro

Processing → OE023 | Norreport

Processing → OE023 | Nordhavn

Processing → OE004 | Norreport

Processing → OE010 | Nordhavn

DONE


In [46]:
output_dir = "/mnt/raid/emotional_data_raquel/typology_landuse_maps"

os.makedirs(output_dir, exist_ok=True)

# =====================================================
# LOOP THROUGH SESSIONS
# =====================================================

for (participant, session), df_sess in df_all.groupby(
    ["participant_id", "session_id"]
):

    print(f"\nPlotting → {participant} | {session}")

    # --------------------------------------------------
    # COPY
    # --------------------------------------------------
    df_sess = df_sess.copy()

    # ==================================================
    # CHOOSE GPS SOURCE
    # ==================================================

    has_corrected = (
        "longitude_corrected" in df_sess.columns
        and "latitude_corrected" in df_sess.columns
        and df_sess["longitude_corrected"].notna().any()
        and df_sess["latitude_corrected"].notna().any()
    )

    if has_corrected:

        print("✓ Using corrected coordinates")

        lon_col = "longitude_corrected"
        lat_col = "latitude_corrected"

    else:

        print("✓ Using original coordinates")

        lon_col = "original_longitude"
        lat_col = "original_latitude"

    # ==================================================
    # TYPOLOGY COLUMN
    # ==================================================

    if "typology" in df_sess.columns:

        typology_col = "typology"

    elif "typology_raquel" in df_sess.columns:

        typology_col = "typology_raquel"

    else:

        print("⚠️ No typology column found")

        continue

    # ==================================================
    # REMOVE INVALID GPS
    # ==================================================

    df_sess = df_sess.dropna(
        subset=[lon_col, lat_col]
    )

    if len(df_sess) == 0:

        print("⚠️ No valid GPS rows")

        continue

    # ==================================================
    # REMOVE DUPLICATES
    # ==================================================

    df_sess = df_sess.drop_duplicates(
        subset=[
            lon_col,
            lat_col,
            typology_col
        ]
    )

    # ==================================================
    # CREATE GEODATAFRAME
    # ==================================================

    gdf = gpd.GeoDataFrame(
        df_sess,
        geometry=gpd.points_from_xy(
            df_sess[lon_col],
            df_sess[lat_col]
        ),
        crs="EPSG:4326"
    )

    # ==================================================
    # TYPOLOGY ORDERING
    # ==================================================

    gdf["typologies"] = pd.Categorical(
        gdf[typology_col],
        categories=typology_order,
        ordered=True
    )

    # ==================================================
    # PROJECT
    # ==================================================

    gdf_web = gdf.to_crs(epsg=3857)

    # --------------------------------------------------
    # SMALLER LANDUSE SUBSET
    # --------------------------------------------------

    bounds = (
        gdf
        .to_crs(25832)
        .buffer(150)
        .total_bounds
    )

    landuse_small = landuse.cx[
        bounds[0]:bounds[2],
        bounds[1]:bounds[3]
    ]

    landuse_web = landuse_small.to_crs(
        epsg=3857
    )

    landuse_web["plot_color"] = (
        landuse_web["landuse_en"]
        .map(landuse_colors)
        .fillna("#cccccc")
    )

    # ==================================================
    # FIGURE
    # ==================================================

    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    # --------------------------------------------------
    # LANDUSE
    # --------------------------------------------------

    landuse_web.plot(
        ax=ax,
        color=landuse_web["plot_color"],
        alpha=0.20,
        linewidth=0,
        zorder=1
    )

    # --------------------------------------------------
    # TYPOLOGIES
    # --------------------------------------------------

    gdf_web.plot(
        ax=ax,
        column="typologies",
        cmap=cmap,
        linewidth=1,
        alpha=0.95,
        legend=False,
        zorder=3
    )

    # ==================================================
    # ZOOM
    # ==================================================

    xmin, ymin, xmax, ymax = (
        gdf_web.total_bounds
    )

    padding = 100

    ax.set_xlim(
        xmin - padding,
        xmax + padding
    )

    ax.set_ylim(
        ymin - padding,
        ymax + padding
    )

    # ==================================================
    # BASEMAP
    # ==================================================

    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        alpha=0.45,
        zorder=0
    )

    # ==================================================
    # LANDUSE LEGEND POSITION
    # ==================================================

    if session.lower() == "nordhavn":

        landuse_loc = "upper left"

    else:

        landuse_loc = "upper right"

    # ==================================================
    # LANDUSE LEGEND
    # ==================================================

    landuse_patches = [

        mpatches.Patch(
            color=color,
            label=label,
            alpha=0.5
        )

        for label, color
        in landuse_colors.items()
    ]

    legend_landuse = ax.legend(
        handles=landuse_patches,
        title="Municipality zones",
        loc=landuse_loc,
        fontsize=8,
        frameon=True
    )

    ax.add_artist(
        legend_landuse
    )

    # ==================================================
    # TYPOLOGY LEGEND
    # ==================================================

    typology_lines = [

        Line2D(
            [0],
            [0],
            color=color,
            lw=4,
            label=label
        )

        for label, color
        in typology_colors.items()
    ]

    legend_typology = ax.legend(
        handles=typology_lines,
        title="Participant typologies",
        loc="lower right",
        fontsize=8,
        frameon=True
    )

    ax.add_artist(
        legend_typology
    )

    # ==================================================
    # TITLE
    # ==================================================

    ax.set_title(
        f"Participant {participant} - {session}",
        fontsize=16,
        pad=10
    )

    ax.set_xticks([])
    ax.set_yticks([])

    # ==================================================
    # SAVE
    # ==================================================

    save_path = os.path.join(
        output_dir,
        f"sub-{participant}_ses-{session}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"✓ Saved → {save_path}")

print("\nDONE")

# OE023 | Norrebro
# OE015 | Norreport
# OE010 | Nordhavn
# OE002 | Hellerup  


Plotting → OE005 | Nordhavn
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE005_ses-Nordhavn.png

Plotting → OE009 | Norrebro
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE009_ses-Norrebro.png

Plotting → OE011 | Nordhavn
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE011_ses-Nordhavn.png

Plotting → OE020 | Norrebro
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE020_ses-Norrebro.png

Plotting → OE021 | Hellerup
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE021_ses-Hellerup.png

Plotting → OE021 | Norrebro
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typology_landuse_maps/sub-OE021_ses-Norrebro.png

Plotting → OE022 | Norrebro
✓ Using original coordinates
✓ Saved → /mnt/raid/emotional_data_raquel/typolo

In [27]:
# Same as above but for sessions also that dont have typologies (not really needed right?)
full_data_root = "/mnt/raid/emotional_data_raquel/fulldata_mine"
output_dir = "/mnt/raid/emotional_data_raquel/full_typology_landuse_maps"

os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# FIND ALL AVAILABLE DATA FILES
# --------------------------------------------------
all_files = []

for participant in os.listdir(full_data_root):

    participant_path = os.path.join(
        full_data_root,
        participant
    )

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(
            participant_path,
            session
        )

        if not os.path.isdir(session_path):
            continue

        # priority:
        # 1. alldata_final.csv
        # 2. alldata.csv

        final_path = os.path.join(
            session_path,
            "alldata_final.csv"
        )

        raw_path = os.path.join(
            session_path,
            "alldata.csv"
        )

        if os.path.exists(final_path):

            all_files.append(final_path)

        elif os.path.exists(raw_path):

            all_files.append(raw_path)

print(f"\nFound {len(all_files)} files\n")

# --------------------------------------------------
# LOOP THROUGH FILES
# --------------------------------------------------
for file_path in all_files:

    participant = os.path.basename(
        os.path.dirname(
            os.path.dirname(file_path)
        )
    )

    session = os.path.basename(
        os.path.dirname(file_path)
    )

    print(f"\nPlotting → {participant} | {session}")

    # --------------------------------------------------
    # LOAD
    # --------------------------------------------------
    df_sess = pd.read_csv(file_path)

    # --------------------------------------------------
    # CHOOSE COORDINATES
    # --------------------------------------------------
    if (
        "longitude_corrected" in df_sess.columns and
        "latitude_corrected" in df_sess.columns and
        df_sess["longitude_corrected"].notna().any() and
        df_sess["latitude_corrected"].notna().any()
    ):

        lon_col = "longitude_corrected"
        lat_col = "latitude_corrected"

        print("  Using corrected coordinates")

    elif (
        "original_longitude" in df_sess.columns and
        "original_latitude" in df_sess.columns and
        df_sess["original_longitude"].notna().any() and
        df_sess["original_latitude"].notna().any()
    ):

        lon_col = "original_longitude"
        lat_col = "original_latitude"

        print("  Using original coordinates")

    else:

        print("  No valid coordinates found → skipping")
        continue

    # --------------------------------------------------
    # REMOVE MISSING COORDINATES
    # --------------------------------------------------
    df_sess = df_sess.dropna(
        subset=[lon_col, lat_col]
    )

    # --------------------------------------------------
    # TYPOLOGY CHECK
    # --------------------------------------------------
    has_typology = (
        "typology" in df_sess.columns and
        df_sess["typology"].notna().any()
    )

    # --------------------------------------------------
    # REMOVE DUPLICATES
    # --------------------------------------------------
    duplicate_subset = [lon_col, lat_col]

    if has_typology:
        duplicate_subset.append("typology")

    df_sess = df_sess.drop_duplicates(
        subset=duplicate_subset
    )

    # --------------------------------------------------
    # GEO DATAFRAME
    # --------------------------------------------------
    gdf = gpd.GeoDataFrame(
        df_sess,
        geometry=gpd.points_from_xy(
            df_sess[lon_col],
            df_sess[lat_col]
        ),
        crs="EPSG:4326"
    )

    # --------------------------------------------------
    # TYPOLOGY CATEGORIES
    # --------------------------------------------------
    if has_typology:

        gdf["typologies"] = pd.Categorical(
            gdf["typology"],
            categories=typology_order,
            ordered=True
        )

    # --------------------------------------------------
    # PROJECT TO WEB MERCATOR
    # --------------------------------------------------
    gdf_web = gdf.to_crs(epsg=3857)

    # --------------------------------------------------
    # SMALLER LANDUSE SUBSET
    # --------------------------------------------------
    bounds = gdf.to_crs(25832).buffer(150).total_bounds

    landuse_small = landuse.cx[
        bounds[0]:bounds[2],
        bounds[1]:bounds[3]
    ]

    landuse_web = landuse_small.to_crs(epsg=3857)

    # --------------------------------------------------
    # LANDUSE COLORS
    # --------------------------------------------------
    landuse_web["plot_color"] = (
        landuse_web["landuse_en"]
        .map(landuse_colors)
        .fillna("#cccccc")
    )

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 12))

    # --------------------------------------------------
    # LANDUSE POLYGONS
    # --------------------------------------------------
    landuse_web.plot(
        ax=ax,
        color=landuse_web["plot_color"],
        alpha=0.20,
        linewidth=0,
        zorder=1
    )

    # --------------------------------------------------
    # PARTICIPANT DATA
    # --------------------------------------------------
    if has_typology:

        gdf_web.plot(
            ax=ax,
            column="typologies",
            cmap=cmap,
            linewidth=1,
            alpha=0.95,
            legend=False,
            zorder=3
        )

    else:

        gdf_web.plot(
            ax=ax,
            color="black",
            markersize=5,
            alpha=0.7,
            zorder=3
        )

    # --------------------------------------------------
    # ZOOM
    # --------------------------------------------------
    xmin, ymin, xmax, ymax = gdf_web.total_bounds

    padding = 100

    ax.set_xlim(xmin - padding, xmax + padding)
    ax.set_ylim(ymin - padding, ymax + padding)

    # --------------------------------------------------
    # BASEMAP
    # --------------------------------------------------
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        alpha=0.45,
        zorder=0
    )

    # --------------------------------------------------
    # LANDUSE LEGEND POSITION
    # --------------------------------------------------
    if session.lower() == "ses-nordhavn":
        landuse_loc = "upper left"
    else:
        landuse_loc = "upper right"

    # --------------------------------------------------
    # LANDUSE LEGEND
    # --------------------------------------------------
    landuse_patches = [
        mpatches.Patch(
            color=color,
            label=label,
            alpha=0.5
        )
        for label, color in landuse_colors.items()
    ]

    legend_landuse = ax.legend(
        handles=landuse_patches,
        title="Municipality zones",
        loc=landuse_loc,
        fontsize=8,
        frameon=True
    )

    ax.add_artist(legend_landuse)

    # --------------------------------------------------
    # TYPOLOGY LEGEND
    # --------------------------------------------------
    if has_typology:

        typology_lines = [
            Line2D(
                [0],
                [0],
                color=color,
                lw=4,
                label=label
            )
            for label, color in typology_colors.items()
        ]

        legend_typology = ax.legend(
            handles=typology_lines,
            title="Participant typologies",
            loc="lower right",
            fontsize=8,
            frameon=True
        )

        ax.add_artist(legend_typology)

    # --------------------------------------------------
    # TITLE
    # --------------------------------------------------
    ax.set_title(
        f"{participant} | {session}",
        fontsize=16,
        pad=10
    )

    ax.set_xticks([])
    ax.set_yticks([])

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    save_path = os.path.join(
        output_dir,
        f"{participant}_{session}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("  Saved successfully")

print("\n✓ ALL MAPS CREATED")


Found 25 files


Plotting → sub-OE020 | ses-Norrebro
  Using original coordinates
  Saved successfully

Plotting → sub-OE020 | ses-Nordhavn
  Using corrected coordinates
  Saved successfully

Plotting → sub-OE005 | ses-Nordhavn
  Using original coordinates
  Saved successfully

Plotting → sub-OE005 | ses-Hellerup
  Using corrected coordinates
  Saved successfully

Plotting → sub-OE018 | ses-Hellerup
  Using corrected coordinates
  Saved successfully

Plotting → sub-OE012 | ses-Norreport
  No valid coordinates found → skipping

Plotting → sub-OE021 | ses-Norrebro
  Using original coordinates
  Saved successfully

Plotting → sub-OE021 | ses-Hellerup
  Using original coordinates
  Saved successfully

Plotting → sub-OE015 | ses-Norreport
  Using corrected coordinates
  Saved successfully

Plotting → sub-OE002 | ses-Hellerup
  Using corrected coordinates
  Saved successfully

Plotting → sub-OE022 | ses-Norrebro
  Using original coordinates
  Saved successfully

Plotting → sub-OE022 | ses-N